# 03 · Building the Dashboard

**Lesson:** [Interactive Data Visualization Dashboard](https://programminghistorian.org/en/lessons/interactive-data-visualization-dashboard)

**Tutorial sections covered:**
- Initiating a Dashboard Instance
- Coding the Frontend
- Coding the Backend
- Testing the Dashboard

## 🎯 Goal
Turn the cleaned data into an interactive Dash app with a date picker and two line charts.

## Load the data from notebook 02
`%run` executes notebook 02 so `df_ukr` and `df_rus` exist here.

In [2]:
%run 02_data_retrieval_and_cleaning.ipynb

## Initiating a Dashboard Instance

In [3]:
# Type the code from this section of the lesson here.
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.LITERA])
server = app.server

### 📝 My notes
- Would be interested in exploring other themes

## Coding the Frontend

In [4]:
# Type the code from this section of the lesson here.
app.layout = dbc.Container(
    [   dbc.Row([ # row 1
        dbc.Col([html.H1('US National Television News Coverage of the War in Ukraine')],
        className="text-center mt-3 mb-1")
    ]
    ),
        dbc.Row([ # row 2
            dbc.Label("Select a date range:", className="fw-bold")
    ]),
    
     dbc.Row([ # row 3
              dcc.DatePickerRange(
                id='date-range',
                min_date_allowed=df_ukr['date_col'].min().date(),
                max_date_allowed=df_ukr['date_col'].max().date(),
                initial_visible_month=df_ukr['date_col'].min().date(),
                start_date=df_ukr['date_col'].min().date(),
                end_date=df_ukr['date_col'].max().date()
              )
    ]),

     dbc.Row([ # row 4
              dbc.Col(dcc.Graph(id='line-graph-ukr'), 
                      )
     ]),

    dbc.Row([ # row 5
              dbc.Col(dcc.Graph(id='line-graph-rus'), 
                      )
     ])

    ])

### 📝 My notes
- I like the analogy about thinking of the dashboard layout as a grid of rows and columns.

## Coding the Backend

In [5]:
# Type the code from this section of the lesson here.

# callback decorator
@app.callback(
    Output('line-graph-ukr', 'figure'),
    Output('line-graph-rus', 'figure'),
    Input('date-range', 'start_date'),
    Input('date-range', 'end_date')   
)

# callback function
def update_output(start_date, end_date):
    # filter dataframes based on updated data range
    mask_ukr = (df_ukr['date_col'] >= start_date) & (df_ukr['date_col'] <= end_date)
    mask_rus = (df_rus['date_col'] >= start_date) & (df_rus['date_col'] <= end_date)
    df_ukr_filtered = df_ukr.loc[mask_ukr]
    df_rus_filtered = df_rus.loc[mask_rus]
    
    # create line graphs based on filtered dataframes
    line_fig_ukr = px.line(df_ukr_filtered, x="date_col", y="Value", 
                     color='Series', title="Coverage of Ukrainian Keywords")
    line_fig_rus = px.line(df_rus_filtered, x='date_col', y='Value', 
                     color='Series', title="Coverage of Russian Keywords")

    # set x-axis title and y-axis title in line graphs 
    line_fig_ukr.update_layout(
                   xaxis_title='Date',
                   yaxis_title='Percentage of Airtime')
    line_fig_rus.update_layout(
                   xaxis_title='Date',
                   yaxis_title='Percentage of Airtime')
    
    # set label format on x-axis in line graphs 
    line_fig_ukr.update_xaxes(tickformat="%b %d<br>%Y")
    line_fig_rus.update_xaxes(tickformat="%b %d<br>%Y")
    
    return line_fig_ukr, line_fig_rus

### 📝 My notes
- This helps define how interactions occur on the dashboard. For this specific graph, it makes changes to what dates are called and then changes the dates being shown on the dashboard. 

## Testing the Dashboard
The lesson runs `app.run(debug=True)` from a `.py` file. Inside Jupyter, use inline mode
to see the dashboard right under the cell (or `jupyter_mode="tab"` to open a browser tab).

In [6]:
app.run(jupyter_mode="inline", debug=True)

### 📝 My notes
- Very cool. I like the display and the expression of the graphs. I would be interested in exploring other things to potentially add to this dashboard. 

## Export to `app.py`
When the dashboard works, paste the **complete** code (imports → data → cleaning → app →
layout → callback) into the cell below and run it. It writes `tutorial_app/app.py`, which is
what Render deploys. Keep the `if __name__ == ...` block at the bottom.

In [8]:
%%writefile ../tutorial_app/app.py
# Standalone tutorial dashboard for Render (combines notebooks 02 and 03).
# Run locally with:  python app.py   — on Render: gunicorn app:server

# --- Imports (notebook 02) ---
import requests
import pandas as pd
from io import StringIO
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px

# --- Data retrieval (notebook 02) ---
start_day_str = '20211228'
last_day_str = '20221231'

query_url_ukr = f"https://api.gdeltproject.org/api/v2/tv/tv?query=(ukraine%20OR%20ukrainian%20OR%20zelenskyy%20OR%20zelensky%20OR%20kiev%20OR%20kyiv)%20market:%22National%22&mode=timelinevol&format=html&datanorm=perc&format=csv&timelinesmooth=5&datacomb=sep&timezoom=yes&STARTDATETIME={start_day_str}120000&ENDDATETIME={last_day_str}120000"

query_url_rus = f"https://api.gdeltproject.org/api/v2/tv/tv?query=(kremlin%20OR%20russia%20OR%20putin%20OR%20moscow%20OR%20russian)%20market:%22National%22&mode=timelinevol&format=html&datanorm=perc&format=csv&timelinesmooth=5&datacomb=sep&timezoom=yes&STARTDATETIME={start_day_str}120000&ENDDATETIME={last_day_str}120000"

def to_df(queryurl):
  response = requests.get(queryurl)
  content_text = StringIO(response.content.decode('utf-8'))
  df = pd.read_csv(content_text)
  return df

df_ukr = to_df(query_url_ukr)
df_rus = to_df(query_url_rus)

# --- Cleaning (notebook 02) ---
df_ukr = df_ukr.rename(columns={df_ukr.columns[0]: "date_col"})
df_rus = df_rus.rename(columns={df_rus.columns[0]: "date_col"})

df_ukr['date_col'] = pd.to_datetime(df_ukr['date_col'])
df_rus['date_col'] = pd.to_datetime(df_rus['date_col'])

df_rus = df_rus[[x in ['CNN', 'FOXNEWS', 'MSNBC'] for x in df_rus.Series]]
df_ukr = df_ukr[[x in ['CNN', 'FOXNEWS', 'MSNBC'] for x in df_ukr.Series]]

# --- App instance (notebook 03) ---
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.LITERA])
server = app.server

# --- Frontend (notebook 03) ---
app.layout = dbc.Container(
    [   dbc.Row([ # row 1
        dbc.Col([html.H1('US National Television News Coverage of the War in Ukraine')],
        className="text-center mt-3 mb-1")
    ]
    ),
        dbc.Row([ # row 2
            dbc.Label("Select a date range:", className="fw-bold")
    ]),

     dbc.Row([ # row 3
              dcc.DatePickerRange(
                id='date-range',
                min_date_allowed=df_ukr['date_col'].min().date(),
                max_date_allowed=df_ukr['date_col'].max().date(),
                initial_visible_month=df_ukr['date_col'].min().date(),
                start_date=df_ukr['date_col'].min().date(),
                end_date=df_ukr['date_col'].max().date()
              )
    ]),

     dbc.Row([ # row 4
              dbc.Col(dcc.Graph(id='line-graph-ukr'),
                      )
     ]),

    dbc.Row([ # row 5
              dbc.Col(dcc.Graph(id='line-graph-rus'),
                      )
     ])

    ])

# --- Backend (notebook 03) ---

# callback decorator
@app.callback(
    Output('line-graph-ukr', 'figure'),
    Output('line-graph-rus', 'figure'),
    Input('date-range', 'start_date'),
    Input('date-range', 'end_date')
)

# callback function
def update_output(start_date, end_date):
    # filter dataframes based on updated data range
    mask_ukr = (df_ukr['date_col'] >= start_date) & (df_ukr['date_col'] <= end_date)
    mask_rus = (df_rus['date_col'] >= start_date) & (df_rus['date_col'] <= end_date)
    df_ukr_filtered = df_ukr.loc[mask_ukr]
    df_rus_filtered = df_rus.loc[mask_rus]

    # create line graphs based on filtered dataframes
    line_fig_ukr = px.line(df_ukr_filtered, x="date_col", y="Value",
                     color='Series', title="Coverage of Ukrainian Keywords")
    line_fig_rus = px.line(df_rus_filtered, x='date_col', y='Value',
                     color='Series', title="Coverage of Russian Keywords")

    # set x-axis title and y-axis title in line graphs
    line_fig_ukr.update_layout(
                   xaxis_title='Date',
                   yaxis_title='Percentage of Airtime')
    line_fig_rus.update_layout(
                   xaxis_title='Date',
                   yaxis_title='Percentage of Airtime')

    # set label format on x-axis in line graphs
    line_fig_ukr.update_xaxes(tickformat="%b %d<br>%Y")
    line_fig_rus.update_xaxes(tickformat="%b %d<br>%Y")

    return line_fig_ukr, line_fig_rus

if __name__ == '__main__':
    app.run(debug=True)

Overwriting ../tutorial_app/app.py


## ⚠️ Problems & fixes
| Problem / error | What fixed it |
|---|---|
|  |  |

## ✅ Takeaways
*What would I reuse for my own dashboard?*
- I am not sure. I need to think about that. 